# t_04：前 200 名 ticker 與人工排除審核

## 排名定義與操作方式
- **本次假設：**以 ticker 出現的「不同 thread 數」降冪排名，同分按 ticker 字母排序。不是留言數、字詞出現次數或 degree。
- 使用 03 的 `threads.parquet`；每個 thread 內的 ticker 先去重，不同檔案的留言已由 03 彙整。分批讀取，不載入全部留言。
- 預設從成功執行紀錄取得 03 路徑；若紀錄不存在，使用 `03_thread_construction` 下時間名稱最新、摘要標記 complete 的 run，並顯示實際來源。也可在設定 cell 明確指定。
- 輸出 `top200_ticker_review.csv`，欄位 `exclude` 預設為 0；人工判定需排除時改為 1，並在 `reason` 填寫理由。不要刪除表格列或修改 ticker。
- 普通英文詞可能同時是真實 ticker，高排名本身不是誤配證據。建議理由區分「普通詞歧義」「非目標證券」等；本 notebook 不自動做研究排除。
- 編輯 CSV 並存檔後，將最後一個 code cell 的 `EXPORT_REVIEW` 改成 `True`，**只重跑最後一個 cell**。若重啟 kernel，將 `REVIEW_FILE` 明確指定為已編輯 CSV 的路徑即可單獨執行該 cell。
- 匯出 `manual_excluded_tickers.txt` 與可追溯的審核摘要；不修改 03、原始 CSV 或現行字典。之後應在 matching 使用的字典扣除這份清單，再重跑 02、03，才能重新判定失去全部 ticker 的留言。此步驟不直接刪除整個 thread。
- 每次重新執行前面的建立 cell 會建立新的審核目錄，舊審核表仍保留，不會覆蓋人工修改。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Data/process').is_dir())
THREAD_RUN=None  # 可指定 03 run 資料夾的完整路徑。
pointer=ROOT/'Data/process/pipeline_runs.json'
if THREAD_RUN is None and pointer.exists():
    reference=json.loads(pointer.read_text(encoding='utf-8')).get('thread_manifest')
    if reference:
        ref=Path(reference)
        if not ref.is_absolute(): ref=ROOT/ref
        THREAD_RUN=ref.parent
if THREAD_RUN is None:
    candidates=[]
    for path in (ROOT/'Data/process/03_thread_construction').iterdir():
        summary_path=path/'run_summary.json'
        if path.is_dir() and summary_path.is_file() and (path/'threads.parquet').is_file():
            if json.loads(summary_path.read_text(encoding='utf-8')).get('status')=='complete': candidates.append(path)
    if not candidates: raise FileNotFoundError('找不到成功完成的 03 run，請指定 THREAD_RUN。')
    THREAD_RUN=max(candidates,key=lambda p:p.name)
THREAD_RUN=Path(THREAD_RUN)
SOURCE=THREAD_RUN/'threads.parquet'
summary_path=THREAD_RUN/'run_summary.json'
if not summary_path.exists(): summary_path=THREAD_RUN/'manifest.json'
assert json.loads(summary_path.read_text(encoding='utf-8'))['status']=='complete'
source_stat=(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
OUT=ROOT/'Data/process/t_04_ticker_review'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True,exist_ok=False)
print('03 來源：',THREAD_RUN)
print('審核輸出：',OUT)


03 來源： C:\Users\eagle\RedditStock\Data\process\03_thread_construction\2026-09-22_03-56-59_UTC
審核輸出： C:\Users\eagle\RedditStock\Data\process\t_04_ticker_review\2026-09-22_09-42-10_UTC


## 分批統計與前 200 名
比例的分母為全部有效 thread 數。日期範圍為 first_timestamp 的 UTC 範圍。作者資訊不在此投影中，以 N/A 表示。

In [2]:
frequency=Counter()
thread_count=0
previous=None
date_min=date_max=None
for batch_no,batch in enumerate(pq.ParquetFile(SOURCE).iter_batches(batch_size=5000,columns=['thread_id','first_timestamp','tickers_in_thread']),1):
    dates=[]
    for row in batch.to_pylist():
        thread=row['thread_id']
        assert thread and (previous is None or previous<thread), '03 應為已去重、排序的 thread 表。'
        previous=thread
        frequency.update(set(row['tickers_in_thread']))
        thread_count+=1
        if row['first_timestamp'] is not None: dates.append(row['first_timestamp'].isoformat())
    if dates:
        date_min=min([date_min,min(dates)]) if date_min else min(dates)
        date_max=max([date_max,max(dates)]) if date_max else max(dates)
    print({'batch':batch_no,'row_count':batch.num_rows,'unique_threads':batch.num_rows,
           'date_min':min(dates) if dates else None,'date_max':max(dates) if dates else None,'unique_authors':'N/A'})
assert thread_count==pq.read_metadata(SOURCE).num_rows
ranked=sorted(frequency.items(),key=lambda item:(-item[1],item[0]))[:200]
review=pd.DataFrame([{'rank':i,'ticker':ticker,'thread_count':count,
                      'thread_share':count/thread_count,'exclude':0,'reason':''}
                     for i,(ticker,count) in enumerate(ranked,1)])
REVIEW_FILE=OUT/'top200_ticker_review.csv'
review.to_csv(REVIEW_FILE,index=False,encoding='utf-8-sig')
summary={'status':'complete','source':str(SOURCE),'source_size_mtime':source_stat,
         'ranking_definition':'distinct thread count descending; ticker ascending for ties',
         'thread_count':thread_count,'unique_tickers':len(frequency),'review_rows':len(review),
         'date_min':date_min,'date_max':date_max,'unique_authors':None,
         'candidates':review[['rank','ticker','thread_count','thread_share']].to_dict(orient='records')}
(OUT/'run_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
assert source_stat==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
with pd.option_context('display.max_rows',200): display(review)
print('請編輯：',REVIEW_FILE)


{'batch': 1, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2012-09-17T20:19:16+00:00', 'date_max': '2015-08-11T02:19:13+00:00', 'unique_authors': 'N/A'}
{'batch': 2, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2015-06-27T15:32:41+00:00', 'date_max': '2016-02-01T08:45:52+00:00', 'unique_authors': 'N/A'}


{'batch': 3, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2015-11-05T13:50:23+00:00', 'date_max': '2016-07-19T19:29:06+00:00', 'unique_authors': 'N/A'}
{'batch': 4, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-02-25T17:49:11+00:00', 'date_max': '2016-05-20T17:20:26+00:00', 'unique_authors': 'N/A'}


{'batch': 5, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-05-09T16:51:37+00:00', 'date_max': '2017-02-11T23:37:18+00:00', 'unique_authors': 'N/A'}
{'batch': 6, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-07-14T03:01:21+00:00', 'date_max': '2017-01-25T04:38:10+00:00', 'unique_authors': 'N/A'}
{'batch': 7, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-08-31T14:03:00+00:00', 'date_max': '2017-02-27T15:24:47+00:00', 'unique_authors': 'N/A'}


{'batch': 8, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-10-25T16:17:32+00:00', 'date_max': '2016-12-17T03:09:28+00:00', 'unique_authors': 'N/A'}
{'batch': 9, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2016-12-14T18:11:21+00:00', 'date_max': '2017-03-27T17:47:00+00:00', 'unique_authors': 'N/A'}
{'batch': 10, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2017-02-09T22:06:37+00:00', 'date_max': '2017-04-15T17:48:36+00:00', 'unique_authors': 'N/A'}


{'batch': 11, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2017-04-03T00:08:17+00:00', 'date_max': '2017-09-03T07:49:06+00:00', 'unique_authors': 'N/A'}
{'batch': 12, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2017-06-02T04:46:33+00:00', 'date_max': '2017-08-04T18:53:37+00:00', 'unique_authors': 'N/A'}
{'batch': 13, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2017-07-24T19:16:07+00:00', 'date_max': '2017-09-15T17:07:47+00:00', 'unique_authors': 'N/A'}


...

{'batch': 48, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2020-05-28T17:38:54+00:00', 'date_max': '2020-08-01T01:41:29+00:00', 'unique_authors': 'N/A'}
{'batch': 49, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2020-06-11T23:00:44+00:00', 'date_max': '2020-07-12T18:52:02+00:00', 'unique_authors': 'N/A'}


{'batch': 50, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2020-06-25T04:43:49+00:00', 'date_max': '2020-07-12T21:17:37+00:00', 'unique_authors': 'N/A'}
{'batch': 51, 'row_count': 5000, 'unique_threads': 5000, 'date_min': '2020-07-12T05:37:57+00:00', 'date_max': '2020-07-31T02:35:33+00:00', 'unique_authors': 'N/A'}


{'batch': 52, 'row_count': 3609, 'unique_threads': 3609, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2020-08-06T01:21:55+00:00', 'unique_authors': 'N/A'}


,rank,ticker,thread_count,thread_share,exclude,reason
0,1,YOU,202685,0.783751,0,
1,2,FOR,176048,0.680750,0,
2,3,ARE,142391,0.550603,0,
3,4,ALL,120858,0.467339,0,
4,5,OUT,114703,0.443538,0,
5,6,MORE,110962,0.429072,0,
6,7,NOW,101664,0.393119,0,
7,8,HAS,100678,0.389306,0,
8,9,GOOD,95276,0.368417,0,
9,10,ONE,93341,0.360935,0,


請編輯： C:\Users\eagle\RedditStock\Data\process\t_04_ticker_review\2026-09-22_09-42-10_UTC\top200_ticker_review.csv


## 人工確認後匯出排除清單
將 exclude 設為 1、填寫 reason 並存檔。此 cell 會讀取磁碟上最新 CSV，檢查 ticker、排名與原始計數沒有被修改；不需要重跑前面的統計。未勾選任何項目時，匯出的清單為空。

In [3]:
from pathlib import Path
import json, hashlib
import pandas as pd
from datetime import datetime, timezone

EXPORT_REVIEW=False  # 人工審核完成後才改為 True。
# kernel 重啟後，可取消下一行註解並填入已編輯檔案的完整路徑。
# REVIEW_FILE=Path(r'C:/Users/eagle/RedditStock/Data/process/t_04_ticker_review/時間資料夾/top200_ticker_review.csv')
if EXPORT_REVIEW:
    review_path=Path(REVIEW_FILE)
    run=json.loads((review_path.parent/'run_summary.json').read_text(encoding='utf-8'))
    edited=pd.read_csv(review_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    expected=pd.DataFrame(run['candidates']).set_index('ticker')
    assert set(edited.columns)=={'rank','ticker','thread_count','thread_share','exclude','reason'}
    assert edited.ticker.is_unique and set(edited.ticker)==set(expected.index), '請勿刪除列或修改 ticker。'
    flags=edited.exclude.str.strip()
    assert flags.isin(['0','1']).all(), 'exclude 只接受 0 或 1。'
    for row in edited.itertuples(index=False):
        assert int(row.rank)==int(expected.loc[row.ticker,'rank'])
        assert int(row.thread_count)==int(expected.loc[row.ticker,'thread_count'])
        assert abs(float(row.thread_share)-float(expected.loc[row.ticker,'thread_share']))<1e-12
    excluded=edited.loc[flags.eq('1')].copy()
    assert excluded.reason.str.strip().ne('').all(), '排除的 ticker 請填 reason。'
    target=review_path.parent/'manual_excluded_tickers.txt'
    tickers=sorted(excluded.ticker.tolist())
    target.write_text(''.join(t+'\n' for t in tickers),encoding='utf-8')
    audit={'source':run['source'],'review_file':str(review_path),'review_sha256':hashlib.sha256(review_path.read_bytes()).hexdigest(),
           'exported_at_utc':datetime.now(timezone.utc).isoformat(),'excluded_count':len(tickers),
           'excluded':excluded[['ticker','reason']].to_dict(orient='records'),'applied_to_data':False}
    (review_path.parent/'manual_review_summary.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print('排除清單：',target,'；ticker 數：',len(tickers))
else:
    print('尚未套用人工排除；請先編輯審核 CSV，再啟用此 cell。')


尚未套用人工排除；請先編輯審核 CSV，再啟用此 cell。
